<div align="center">

# Trivia Night with KONASH

**Train a 4B model to answer multi-constraint trivia by searching Wikipedia**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konaequity/openkona/blob/main/notebooks/trivia_night.ipynb)

</div>

---

Run all cells via **Runtime → Run all**. Requires a Colab GPU runtime — go to **Runtime → Change runtime type → T4 GPU** before running.

This notebook trains a **Qwen 3 4B** model to answer multi-constraint trivia questions. The agent learns to search a Wikipedia corpus, retrieve relevant evidence, and synthesize accurate answers. Training takes ~30 minutes on a T4.

---

In [ ]:
!pip install -qU "konash[train] @ git+https://github.com/konaequity/openkona.git"

In [ ]:
import os
import json
import urllib.request
import urllib.parse

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > Select GPU.")

## Agentic Environment

The environment defines the knowledge the agent can search over. We fetch 25 Wikipedia articles spanning history, science, geography, literature, and pop culture — the pillars of any good trivia night — and write them to disk as the agent's searchable corpus.

In [ ]:
def fetch_wikipedia_article(title: str) -> str:
    """Fetch plain-text extract of a Wikipedia article via the API."""
    params = urllib.parse.urlencode({
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": "1",
        "exsectionformat": "plain",
        "format": "json",
    })
    url = f"https://en.wikipedia.org/w/api.php?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": "KONASH-TriviaNotebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    pages = data.get("query", {}).get("pages", {})
    for page in pages.values():
        return page.get("extract", "")
    return ""


TRIVIA_TOPICS = [
    # History
    "Ancient Rome", "French Revolution", "Silk Road",
    "Space Race", "Rosetta Stone",
    # Science
    "Periodic table", "Theory of relativity", "DNA",
    "Black hole", "Photosynthesis",
    # Geography
    "Amazon River", "Mount Everest", "Great Barrier Reef",
    "Sahara", "Ring of Fire",
    # Literature & Arts
    "William Shakespeare", "Leonardo da Vinci",
    "The Great Gatsby", "Odyssey",
    # Pop Culture & Sports
    "Olympic Games", "FIFA World Cup", "Academy Awards",
    "Beatles", "Chess",
    # Miscellaneous
    "Seven Wonders of the Ancient World",
]

CORPUS_DIR = "./trivia_corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

print(f"Fetching {len(TRIVIA_TOPICS)} Wikipedia articles...")
for i, topic in enumerate(TRIVIA_TOPICS):
    text = fetch_wikipedia_article(topic)
    if text and len(text) > 200:
        words = text.split()
        if len(words) > 3000:
            text = " ".join(words[:3000])
        fname = topic.replace(" ", "_").replace("/", "_") + ".txt"
        with open(os.path.join(CORPUS_DIR, fname), "w") as f:
            f.write(text)
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: {len(text):,} chars")
    else:
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: SKIPPED (too short)")

## Create the Agent

We create a `konash.Agent` backed by Qwen 3 4B with 4-bit QLoRA quantization. This fits on a T4 GPU (16 GB VRAM). The agent will automatically chunk, embed, and index the corpus when training begins.

In [ ]:
import konash

agent = konash.Agent(
    base_model="Qwen/Qwen3-4B",
    corpus=CORPUS_DIR,
    project="trivia-night",
    chunk_size=384,
    load_in_4bit=True,
    gradient_checkpointing=True,
)

## Train

Each iteration, the agent synthesizes multi-constraint trivia questions from the corpus, generates search-and-answer rollouts, filters to questions at the learning frontier, and trains with OAPL. Two iterations is enough to see clear improvement.

In [ ]:
results = agent.train(iterations=2, rollouts_per_example=8)

## Play Trivia

The trained agent can now answer trivia questions by searching the Wikipedia corpus. Let's try it on a few questions it has never seen.

In [ ]:
TRIVIA_QUESTIONS = [
    # Requires finding a specific number buried in the French Revolution article
    "How many deputies voted in the National Convention's roll-call vote on the death of Louis XVI, and what was the exact margin by which execution was approved?",
    # Requires cross-referencing specific details from the DNA article
    "What was the designation number of the X-ray diffraction image taken by Rosalind Franklin that provided crucial evidence for DNA's helical structure?",
    # Obscure detail from the Periodic Table article
    "What name did Mendeleev give to his predicted element 'eka-aluminum' before it was discovered, and what is that element's modern name and atomic number?",
    # Specific measurement from the Amazon River article
    "What is the average discharge rate of the Amazon River at its mouth in cubic meters per second, and roughly what fraction of all river water entering the ocean does this represent?",
    # Obscure detail from the Chess article
    "What is the Shannon number — the estimated number of possible chess games — and who calculated it?",
    # Specific detail from the Ring of Fire article
    "What percentage of the world's active and dormant volcanoes are found along the Pacific Ring of Fire, and approximately how many volcanoes does it contain?",
]

for q in TRIVIA_QUESTIONS:
    answer = agent.solve(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

For higher accuracy, use **Parallel Thinking** — the agent runs multiple independent search rollouts and aggregates their answers.

In [ ]:
q = "What specific mathematical proof did Leonardo da Vinci contribute to, involving the doubling of a square's area, and in which of his notebooks does it appear?"
answer = agent.solve(q, parallel_rollouts=5)
print(f"Q: {q}")
print(f"A: {answer}")

---

Your trivia agent is trained and ready! Check out the [KONASH GitHub](https://github.com/konaequity/openkona) for more examples.

**Next steps:**
- Add more Wikipedia articles to the corpus for broader coverage
- Run a 3rd training iteration for further improvement
- Use Parallel Thinking (`parallel_rollouts=10-20`) for near-deterministic accuracy
- Deploy with vLLM + LoRA hot-swapping for production serving